In [1]:
import sys
import os
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
df = pd.read_csv("../data/output/cleaned_reservations.csv")
df.head()

,Reservation_ID,Vehicle_ID,Vehicle_Class,Booking_TS,Pickup_TS,Return_TS,Pickup_Lat,Pickup_Lon,Drop_Lat,Drop_Lon,...,Idle_Time,Fuel_Efficiency,Damage_Rate,Maintenance_Due,Overstay_Minutes,Pickup_Delay,Driver_License,Driver_Score,Fleet_Health_Score,Churn_Risk
0,RES-01325,CAR-01,Luxury,2025-12-31 08:27:00,2026-01-01 21:27:00,2026-01-05 15:27:00,13.091661,80.271829,13.073789,80.282755,...,0.000000,3374.193548,NaN,1,4920.0,0,DL67XXXXXXX,94,60,2.4
1,RES-03673,CAR-01,Compact,2026-01-01 10:57:00,2026-01-03 06:57:00,2026-01-10 21:57:00,13.082735,80.265113,13.083241,80.264692,...,-56.500000,5707.692308,NaN,1,10500.0,0,DL67XXXXXXX,60,40,22.0
2,RES-03288,CAR-01,Economy,2026-01-02 17:34:00,2026-01-03 13:34:00,2026-01-11 07:34:00,22.571409,88.382399,22.571633,88.356736,...,-176.383333,1043.750000,NaN,1,10680.0,0,DL67XXXXXXX,82,60,7.2
3,RES-01238,CAR-01,Economy,2026-01-03 22:44:00,2026-01-03 23:44:00,2026-01-04 18:44:00,28.714020,77.104070,28.691836,77.098341,...,-175.833333,2025.000000,NaN,1,660.0,0,DL12XXXXXXX,90,60,4.0
4,RES-03927,CAR-01,SUV,2026-01-01 22:29:00,2026-01-04 09:29:00,2026-01-06 14:29:00,12.966111,77.589671,12.971982,77.579039,...,-9.250000,3015.094340,NaN,1,2700.0,0,DL90XXXXXXX,90,60,4.0


In [3]:
df.columns

Index(['Reservation_ID', 'Vehicle_ID', 'Vehicle_Class', 'Booking_TS',
       'Pickup_TS', 'Return_TS', 'Pickup_Lat', 'Pickup_Lon', 'Drop_Lat',
       'Drop_Lon', 'Odo_Start', 'Odo_End', 'Fuel_Level', 'Rate', 'City',
       'Payment', 'Fraud_Risk_Score', 'Fraud_Risk_Level', 'Distance_km',
       'Rental_Hours', 'Revenue', 'Cost_per_km', 'Utilization', 'RevPAC',
       'Idle_Time', 'Fuel_Efficiency', 'Damage_Rate', 'Maintenance_Due',
       'Overstay_Minutes', 'Pickup_Delay', 'Driver_License', 'Driver_Score',
       'Fleet_Health_Score', 'Churn_Risk'],
      dtype='str')

In [4]:
sys.path.append(os.path.abspath("../src"))

from analytics.analytics_engine import AnalyticalEngine
engine = AnalyticalEngine(df)


1

In [5]:
df = engine.compute_utilization()

df[["Vehicle_ID","Rental_Hours","Utilization"]].head()


,Vehicle_ID,Rental_Hours,Utilization
0,CAR-01,90.0,3.750000
1,CAR-01,183.0,7.625000
2,CAR-01,186.0,7.750000
3,CAR-01,19.0,0.791667
4,CAR-01,53.0,2.208333


2. Revenue per available car

In [6]:
df = engine.compute_revpac()

df["RevPAC"] = (df["Revenue"] / 24).round(2)

df[["Vehicle_ID","Revenue","RevPAC"]].head()



,Vehicle_ID,Revenue,RevPAC
0,CAR-01,2000,83.33
1,CAR-01,3000,125.00
2,CAR-01,2500,104.17
3,CAR-01,1500,62.50
4,CAR-01,3000,125.00


3. Distance Driven & Cost per KM

In [7]:
df = engine.compute_distance_cost()

df[["Odo_Start","Odo_End","Distance_km","Cost_per_km"]].head()


,Odo_Start,Odo_End,Distance_km,Cost_per_km
0,116111,117157,1046,1.912046
1,49480,50222,742,4.043127
2,106071,106906,835,2.994012
3,31255,31822,567,2.645503
4,101014,102612,1598,1.877347


4. Idle Time Analytics

In [8]:
df["Pickup_TS"] = pd.to_datetime(df["Pickup_TS"])
df["Return_TS"] = pd.to_datetime(df["Return_TS"])

df = df.sort_values(["Vehicle_ID", "Pickup_TS"])

df["Prev_Return"] = df.groupby("Vehicle_ID")["Return_TS"].shift(1)

df["Idle_Time"] = (df["Pickup_TS"] - df["Prev_Return"]).dt.total_seconds() / 3600

df["Idle_Time"] = df["Idle_Time"].fillna(0)


In [9]:

df[["Vehicle_ID","Prev_Return","Pickup_TS","Idle_Time"]].head()


,Vehicle_ID,Prev_Return,Pickup_TS,Idle_Time
0,CAR-01,NaT,2026-01-01 21:27:00,0.000000
1,CAR-01,2026-01-05 15:27:00,2026-01-03 06:57:00,-56.500000
2,CAR-01,2026-01-10 21:57:00,2026-01-03 13:34:00,-176.383333
3,CAR-01,2026-01-11 07:34:00,2026-01-03 23:44:00,-175.833333
4,CAR-01,2026-01-04 18:44:00,2026-01-04 09:29:00,-9.250000


Negative idle times indicate overlapping bookings for the same vehicle. In real fleet analytics pipelines, such values are either flagged as booking conflicts or clamped to zero since idle time cannot be negative.

5
Repositioning Distance

In [10]:
df = engine.repositioning_distance()

df[["Vehicle_ID","Reposition_Distance"]].head()


,Vehicle_ID,Reposition_Distance
0,CAR-01,0.000000
1,CAR-01,0.019781
2,CAR-01,12.486893
3,CAR-01,12.819961
4,CAR-01,15.733399


6
Dynamic Pricing Features

In [11]:
df = engine.dynamic_pricing_features()

df[["City","Demand_Index","Lead_Time","Season"]].head()


,City,Demand_Index,Lead_Time,Season
0,Chennai,598,4,1
1,Chennai,598,8,1
2,Kolkata,580,8,1
3,Delhi,619,1,1
4,Bengaluru,590,3,1


7
Fuel Efficiency

In [12]:
df = engine.fuel_efficiency()

df[["Distance_km","Fuel_Level","Fuel_Efficiency"]].head()


,Distance_km,Fuel_Level,Fuel_Efficiency
0,1046,0.31,3374.193548
1,742,0.13,5707.692308
2,835,0.80,1043.750000
3,567,0.28,2025.000000
4,1598,0.53,3015.094340


8
Damage Incidence Rate

In [13]:
df['Fuel_Level'].describe

<bound method NDFrame.describe of 0       0.31
1       0.13
2       0.80
3       0.28
4       0.53
        ... 
3637    0.60
3638    0.90
3639    0.91
3640    0.75
3641    0.30
Name: Fuel_Level, Length: 3642, dtype: float64>

In [14]:
df = engine.damage_rate()

df[["Fuel_Level","Damage_Flag"]].head()


,Fuel_Level,Damage_Flag
0,0.31,0
1,0.13,1
2,0.80,0
3,0.28,0
4,0.53,0


In [15]:
df = engine.damage_rate()

damage_rate = engine.damage_incidence_rate()

print("Damage Incidence Rate per 100 rentals:", damage_rate)


Damage Incidence Rate per 100 rentals: 13.04


9
Customer Cohort Retention

In [16]:
df.columns

Index(['Reservation_ID', 'Vehicle_ID', 'Vehicle_Class', 'Booking_TS',
       'Pickup_TS', 'Return_TS', 'Pickup_Lat', 'Pickup_Lon', 'Drop_Lat',
       'Drop_Lon', 'Odo_Start', 'Odo_End', 'Fuel_Level', 'Rate', 'City',
       'Payment', 'Fraud_Risk_Score', 'Fraud_Risk_Level', 'Distance_km',
       'Rental_Hours', 'Revenue', 'Cost_per_km', 'Utilization', 'RevPAC',
       'Idle_Time', 'Fuel_Efficiency', 'Damage_Rate', 'Maintenance_Due',
       'Overstay_Minutes', 'Pickup_Delay', 'Driver_License', 'Driver_Score',
       'Fleet_Health_Score', 'Churn_Risk', 'Prev_Drop_Lat', 'Prev_Drop_Lon',
       'Reposition_Distance', 'Demand_Index', 'Lead_Time', 'Season',
       'Damage_Flag'],
      dtype='str')

In [17]:
df = engine.cohort_retention()

df[["Driver_License","Cohort_Month","Activity_Month","Cohort_Index"]].head()


,Driver_License,Cohort_Month,Activity_Month,Cohort_Index
0,DL67XXXXXXX,2026-01,2026-01,0
1,DL67XXXXXXX,2026-01,2026-01,0
2,DL67XXXXXXX,2026-01,2026-01,0
3,DL12XXXXXXX,2026-01,2026-01,0
4,DL90XXXXXXX,2026-01,2026-01,0


In [18]:
df["Cohort_Index"].unique

<bound method Series.unique of 0       0
1       0
2       0
3       0
4       0
       ..
3637    2
3638    3
3639    3
3640    3
3641    3
Name: Cohort_Index, Length: 3642, dtype: int64>

10
Fraud Risk Score

In [19]:
df = engine.fraud_risk()

df[["Fraud_Risk_Score","Fraud_Risk_Level"]].head(10)


,Fraud_Risk_Score,Fraud_Risk_Level
0,80,High
1,80,High
2,80,High
3,10,Low
4,10,Low
5,10,Low
6,80,High
7,80,High
8,80,High
9,80,High


In [20]:
df["Fraud_Risk_Score"].value_counts()


Fraud_Risk_Score
80    2688
10     954
Name: count, dtype: int64

11
Maintenance Due Forecast

In [21]:
df = engine.maintenance_due()

df[["Distance_km","Rental_Hours","Maintenance_Due"]].head()


,Distance_km,Rental_Hours,Maintenance_Due
0,1046,90.0,1
1,742,183.0,1
2,835,186.0,1
3,567,19.0,1
4,1598,53.0,1


12
Overstay Detection

In [22]:
df = engine.overstay_detection()

df[["Rental_Hours","Overstay_Minutes","Overstay_Penalty"]].head()


,Rental_Hours,Overstay_Minutes,Overstay_Penalty
0,90.0,4920.0,820.0
1,183.0,10500.0,1750.0
2,186.0,10680.0,1780.0
3,19.0,660.0,110.0
4,53.0,2700.0,450.0



13
Pickup / Return Punctuality

In [23]:
df = engine.pickup_punctuality()

print("Average Return Delay (minutes):", engine.avg_return_delay)
print("On-Time Return %:", engine.on_time_returns)

df[["Return_Delay_Min"]].head()


Average Return Delay (minutes): 0.0
On-Time Return %: 98.98


,Return_Delay_Min
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0


14
Geo Hotspot Detection

In [24]:
df = engine.geo_hotspots()

df[["Pickup_Lat","Pickup_Lon","Pickup_Hotspot"]].head()


,Pickup_Lat,Pickup_Lon,Pickup_Hotspot
0,13.091661,80.271829,1
1,13.082735,80.265113,1
2,22.571409,88.382399,1
3,28.714020,77.104070,1
4,12.966111,77.589671,1


15
Upsell / Cross-Sell Flags

In [25]:
df = engine.upsell_flags()

df[["Rate","Distance_km","Upsell_Flag"]].head()


,Rate,Distance_km,Upsell_Flag
0,2000,1046,1
1,3000,742,1
2,2500,835,1
3,1500,567,1
4,3000,1598,1


16
Cancellation Rate Analysis

In [26]:
df = engine.cancellation_rate()

df[["Cancelled","Cancellation_Reason"]].head()


,Cancelled,Cancellation_Reason
0,0,None
1,0,None
2,0,None
3,0,None
4,0,None


17
Driver Behavior Scoring

In [27]:
df = engine.driver_behavior()

df[["Harsh_Events","Max_Speed_kmh","Driver_Score"]].head()


,Harsh_Events,Max_Speed_kmh,Driver_Score
0,0,80,100
1,0,80,80
2,0,80,100
3,0,80,100
4,0,80,100


18
Vehicle Class Mix Optimization

In [28]:
vehicle_mix = engine.vehicle_mix()

vehicle_mix

Vehicle_Class,Compact,Economy,Luxury,SUV
City,,,,
Bengaluru,145,137,154,154
Chennai,152,146,146,154
Delhi,170,141,160,148
Hyderabad,156,163,159,153
Kolkata,163,136,140,141
Mumbai,147,162,166,149


19
Lead-Time Price Elasticity

In [29]:
df = engine.price_elasticity()

df[["Lead_Time","Rate","Elasticity_Feature"]].head()

,Lead_Time,Rate,Elasticity_Feature
0,37.0,2000,74000.0
1,44.0,3000,132000.0
2,20.0,2500,50000.0
3,1.0,1500,1500.0
4,59.0,3000,177000.0


20
Churn Likelihood Prediction

In [30]:
df = engine.churn_prediction()

df[["Driver_Score","Pickup_Delay","Churn_Risk"]].head()

,Driver_Score,Pickup_Delay,Churn_Risk
0,100,0,0.0
1,80,0,14.0
2,100,0,0.0
3,100,0,0.0
4,100,0,0.0


In [31]:
df.to_csv("../data/processed/analytics_ready_data.csv", index=False)

print("Analytics dataset saved successfully.")

Analytics dataset saved successfully.
